# Irodori-TTS 音声合成 Colab ノートブック

[Irodori-TTS](https://github.com/Aratako/Irodori-TTS) を使用したローカル日本語音声合成ノートブックです。  
**テキストプロンプトで声質・感情・話し方を自在に指定**できます（参照音声不要）。

## 前提条件
- ランタイムを **T4 GPU** に設定してください（ランタイム → ランタイムのタイプを変更）
- 初回実行時にモデル（約 2GB）を HuggingFace からダウンロードします

## モデル
| チェックポイント | 特徴 |
|---|---|
| `Aratako/Irodori-TTS-500M-v2-VoiceDesign` | テキストキャプションで声質・感情を制御（参照音声不要） |

In [ ]:
# ── セットアップ（初回のみ数分かかります）──────────────────────────────
import subprocess, sys, os

# GPU 確認
res = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if res.returncode == 0:
    print(f"✅ GPU 検出: {res.stdout.strip()}")
else:
    raise RuntimeError("GPU が見つかりません。ランタイムのタイプを T4 GPU に変更してください。")

# Irodori-TTS リポジトリ取得
REPO_DIR = '/content/Irodori-TTS'
if not os.path.exists(REPO_DIR):
    print("\n📥 Irodori-TTS をクローン中...")
    subprocess.run(
        ['git', 'clone', 'https://github.com/Aratako/Irodori-TTS.git', REPO_DIR],
        check=True
    )
    print("✅ クローン完了")
else:
    print(f"\n✅ {REPO_DIR} は既に存在します")

# 依存関係インストール
# （Colab に既存の PyTorch はそのまま利用。git 依存の dacvae / silentcipher を含む）
print("\n📦 依存関係をインストール中（初回は数分かかります）...")
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements.txt'],
    check=True
)
print("✅ インストール完了")

# Python パスにリポジトリを追加
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("\n🎉 セットアップ完了")

In [ ]:
# ── Google Drive マウント ──────────────────────────────────────────────
# このセルを実行すると Google アカウントへの認証が求められます
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_DIR          = '/content/drive/MyDrive/Irodori-TTS'
DRIVE_PRESETS_PATH = f'{DRIVE_DIR}/presets.json'

os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"✅ Google Drive マウント完了")
print(f"📁 プリセット保存先: {DRIVE_PRESETS_PATH}")

In [ ]:
# ── モデル読み込み（セッション中に 1 度だけ実行）──────────────────────
from huggingface_hub import hf_hub_download
from irodori_tts.inference_runtime import RuntimeKey, SamplingRequest, get_cached_runtime

HF_REPO         = "Aratako/Irodori-TTS-500M-v2-VoiceDesign"
MODEL_DEVICE    = "cuda"
CODEC_REPO      = "Aratako/Semantic-DACVAE-Japanese-32dim"
MODEL_PRECISION = "bf16"
CODEC_PRECISION = "bf16"

# RuntimeKey.checkpoint はローカルパスのみ受け付けるため、先に HF からダウンロードする
print(f"📥 モデルをダウンロード中: {HF_REPO}")
print("   初回は HuggingFace からダウンロードします（約 2GB）...\n")
local_checkpoint = hf_hub_download(repo_id=HF_REPO, filename="model.safetensors")
print(f"✅ ダウンロード完了: {local_checkpoint}\n")

print("🔄 モデルを読み込み中...")
runtime_key = RuntimeKey(
    checkpoint=local_checkpoint,
    model_device=MODEL_DEVICE,
    codec_repo=CODEC_REPO,
    model_precision=MODEL_PRECISION,
    codec_device=MODEL_DEVICE,
    codec_precision=CODEC_PRECISION,
    compile_model=False,
    compile_dynamic=False,
)

runtime, _ = get_cached_runtime(runtime_key)
print("\n✅ モデル読み込み完了")

In [ ]:
# ── Gradio UI ─────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lameenc'], check=True)

import gradio as gr
import soundfile as sf
import lameenc
import numpy as np
import json, tempfile, os, random, time

os.makedirs("/content/audio_output", exist_ok=True)

# ── Google Drive プリセット管理 ─────────────────────────────────────

def _drive_path():
    try:
        return DRIVE_PRESETS_PATH
    except NameError:
        return None

def load_presets():
    path = _drive_path()
    if path is None or not os.path.exists(path):
        return []
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def _save_presets(presets):
    path = _drive_path()
    if path is None:
        raise RuntimeError("Google Drive がマウントされていません。Drive マウントセルを実行してください。")
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(presets, f, ensure_ascii=False, indent=2)

def preset_choices():
    return [p['name'] for p in load_presets()]

def save_preset_fn(name, text, caption, num_steps, cfg_text, cfg_caption, seed):
    if not name.strip():
        return gr.Dropdown(choices=preset_choices()), "❌ プリセット名を入力してください"
    try:
        presets = load_presets()
        entry = {
            "name": name.strip(),
            "text": text,
            "caption": caption,
            "num_steps": int(num_steps),
            "cfg_scale_text": float(cfg_text),
            "cfg_scale_caption": float(cfg_caption),
            "seed": int(seed),
            "saved_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
        }
        names = [p['name'] for p in presets]
        if name.strip() in names:
            presets[names.index(name.strip())] = entry
            msg = f"✅ 「{name.strip()}」を更新しました"
        else:
            presets.append(entry)
            msg = f"✅ 「{name.strip()}」を保存しました"
        _save_presets(presets)
        choices = [p['name'] for p in presets]
        return gr.Dropdown(choices=choices, value=name.strip()), msg
    except Exception as e:
        return gr.Dropdown(choices=preset_choices()), f"❌ {e}"

def load_preset_fn(name):
    _u = gr.update()
    if not name:
        return _u, _u, _u, _u, _u, _u, "❌ プリセットを選択してください"
    try:
        preset = next((p for p in load_presets() if p['name'] == name), None)
        if not preset:
            return _u, _u, _u, _u, _u, _u, f"❌ 「{name}」が見つかりません"
        return (
            preset.get('text', ''),
            preset.get('caption', ''),
            preset.get('num_steps', 40),
            preset.get('cfg_scale_text', 2.0),
            preset.get('cfg_scale_caption', 2.0),
            preset.get('seed', 42),
            f"✅ 「{name}」を読み込みました",
        )
    except Exception as e:
        return _u, _u, _u, _u, _u, _u, f"❌ {e}"

def delete_preset_fn(name):
    if not name:
        return gr.Dropdown(choices=preset_choices()), "❌ プリセットを選択してください"
    try:
        presets = [p for p in load_presets() if p['name'] != name]
        _save_presets(presets)
        choices = [p['name'] for p in presets]
        return gr.Dropdown(choices=choices, value=choices[0] if choices else None), f"✅ 「{name}」を削除しました"
    except Exception as e:
        return gr.Dropdown(choices=preset_choices()), f"❌ {e}"

def refresh_presets_fn():
    choices = preset_choices()
    return gr.Dropdown(choices=choices, value=choices[0] if choices else None), f"🔄 {len(choices)} 件のプリセットを読み込みました"

# ── 音声合成 ────────────────────────────────────────────────────────

PRESET_CAPTIONS = [
    ("落ち着いた女性",  "落ち着いた女性の声で、近い距離感でやわらかく自然に読み上げてください。"),
    ("明るい女性",      "明るく元気な若い女性の声で、笑顔が伝わるように読み上げてください。😊"),
    ("落ち着いた男性",  "落ち着いた中年男性の声で、ゆっくりと丁寧に読み上げてください。"),
    ("疲れた声",        "少し疲れた様子で、感情を込めてつぶやくように読み上げてください。😔"),
    ("アナウンサー",    "NHKのアナウンサーのように、明瞭で標準的な発音で読み上げてください。"),
]

def save_mp3(audio_np: np.ndarray, sample_rate: int, path: str) -> None:
    mono = audio_np if audio_np.ndim == 1 else audio_np[:, 0]
    pcm = (mono * 32767).clip(-32768, 32767).astype(np.int16)
    enc = lameenc.Encoder()
    enc.set_bit_rate(192)
    enc.set_in_sample_rate(sample_rate)
    enc.set_channels(1)
    enc.set_quality(2)
    data = enc.encode(pcm.tobytes()) + enc.flush()
    with open(path, "wb") as f:
        f.write(data)

def synthesize_audio(text, caption, num_steps, cfg_text, cfg_caption, seed):
    hidden = gr.DownloadButton(visible=False)
    if not text.strip():
        return None, "❌ テキストを入力してください", hidden, hidden

    actual_seed = random.randint(0, 2**31 - 1) if int(seed) < 0 else int(seed)

    try:
        req = SamplingRequest(
            text=text,
            caption=caption.strip() or None,
            no_ref=True,
            num_steps=int(num_steps),
            seed=actual_seed,
            cfg_scale_text=float(cfg_text),
            cfg_scale_caption=float(cfg_caption),
        )
        result = runtime.synthesize(req)
    except Exception as e:
        return None, f"❌ エラー: {e}", hidden, hidden

    audio_np = result.audio.cpu().float().numpy()
    if audio_np.ndim == 2:
        audio_np = audio_np.T

    base = tempfile.mktemp(dir="/content/audio_output")
    wav_path = base + ".wav"
    mp3_path = base + ".mp3"
    sf.write(wav_path, audio_np, result.sample_rate)
    save_mp3(audio_np, result.sample_rate, mp3_path)

    status = f"✅ 完了（シード: {actual_seed}、サンプルレート: {result.sample_rate} Hz）"
    return (
        wav_path,
        status,
        gr.DownloadButton(value=wav_path, visible=True),
        gr.DownloadButton(value=mp3_path, visible=True),
    )

# ── UI 構築 ─────────────────────────────────────────────────────────

with gr.Blocks(title="Irodori-TTS", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ Irodori-TTS 音声合成")

    with gr.Row():
        with gr.Column(scale=3):
            text_input = gr.Textbox(
                label="読み上げテキスト",
                placeholder="ここに読み上げるテキストを入力してください",
                lines=5,
                value="こんにちは。今日はどんなお話をしましょうか？もしよければ、ゆっくりお話ししますね。",
            )
            caption_input = gr.Textbox(
                label="声質・感情プロンプト（キャプション）",
                placeholder="声の雰囲気・感情・話し方を日本語・英語・絵文字で記述",
                lines=3,
                value="落ち着いた女性の声で、近い距離感でやわらかく自然に読み上げてください。",
            )
            gr.Markdown("**キャプションプリセット**")
            with gr.Row():
                for name, cap in PRESET_CAPTIONS:
                    gr.Button(name, size="sm").click(
                        fn=lambda c=cap: c,
                        outputs=caption_input,
                    )

        with gr.Column(scale=2):
            num_steps = gr.Slider(20, 80, value=40, step=1,
                                  label="ステップ数（多いほど高品質・低速）")
            cfg_text = gr.Slider(0.5, 5.0, value=2.0, step=0.1,
                                 label="CFG スケール（テキスト）")
            cfg_caption = gr.Slider(0.5, 5.0, value=2.0, step=0.1,
                                    label="CFG スケール（キャプション）")
            seed = gr.Number(value=42, precision=0,
                             label="シード（-1 でランダム）")

    generate_btn = gr.Button("🎙️ 合成する", variant="primary", size="lg")
    audio_output = gr.Audio(label="出力音声", type="filepath")
    status_text  = gr.Textbox(label="ステータス", interactive=False, lines=1)
    with gr.Row():
        download_wav = gr.DownloadButton("⬇️ WAV をダウンロード",
                                         visible=False, variant="primary", size="lg")
        download_mp3 = gr.DownloadButton("⬇️ MP3 をダウンロード",
                                         visible=False, variant="secondary", size="lg")

    with gr.Accordion("📋 プリセット管理（Google Drive）", open=False):
        gr.Markdown("現在の入力・設定を名前付きで Google Drive に保存／読み込みできます。")

        with gr.Row():
            preset_name_input = gr.Textbox(
                label="プリセット名",
                placeholder="例：落ち着いた女性・低速",
                scale=3,
            )
            save_preset_btn   = gr.Button("💾 保存", variant="primary", scale=1)
            delete_preset_btn = gr.Button("🗑 削除", variant="stop",    scale=1)

        with gr.Row():
            preset_dropdown = gr.Dropdown(
                label="保存済みプリセット",
                choices=preset_choices(),
                value=None,
                scale=3,
            )
            load_preset_btn = gr.Button("📂 読み込む", scale=1)
            refresh_btn     = gr.Button("🔄 更新",     scale=1)

        preset_status = gr.Textbox(label="ステータス", interactive=False, lines=1)

    generate_btn.click(
        fn=synthesize_audio,
        inputs=[text_input, caption_input, num_steps, cfg_text, cfg_caption, seed],
        outputs=[audio_output, status_text, download_wav, download_mp3],
    )
    save_preset_btn.click(
        fn=save_preset_fn,
        inputs=[preset_name_input, text_input, caption_input,
                num_steps, cfg_text, cfg_caption, seed],
        outputs=[preset_dropdown, preset_status],
    )
    load_preset_btn.click(
        fn=load_preset_fn,
        inputs=[preset_dropdown],
        outputs=[text_input, caption_input, num_steps, cfg_text, cfg_caption,
                 seed, preset_status],
    )
    delete_preset_btn.click(
        fn=delete_preset_fn,
        inputs=[preset_dropdown],
        outputs=[preset_dropdown, preset_status],
    )
    refresh_btn.click(
        fn=refresh_presets_fn,
        outputs=[preset_dropdown, preset_status],
    )

demo.launch(share=True, debug=False)

In [ ]:
# ── 音声合成（基本）────────────────────────────────────────────────────
import datetime, os
import soundfile as sf
from IPython.display import Audio, display

LOCAL_AUDIO_DIR = "/content/audio_output"
os.makedirs(LOCAL_AUDIO_DIR, exist_ok=True)

# ── 読み上げテキスト ────────────────────────────────────────────────
TEXT = "こんにちは。今日はどんなお話をしましょうか？もしよければ、ゆっくりお話ししますね。"

# ── 声質・感情プロンプト（VoiceDesign キャプション）──────────────────
# 日本語・英語・絵文字で声の雰囲気、感情、話し方を自由に記述します
CAPTION = "落ち着いた女性の声で、近い距離感でやわらかく自然に読み上げてください。"

# ── 合成パラメータ ──────────────────────────────────────────────────
NUM_STEPS         = 40    # 拡散ステップ数（多いほど高品質・低速。20〜60 が実用域）
CFG_SCALE_TEXT    = 2.0   # テキスト誘導強度
CFG_SCALE_CAPTION = 2.0   # キャプション誘導強度
SEED              = 42

# ── 合成実行 ────────────────────────────────────────────────────────
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = f"{LOCAL_AUDIO_DIR}/irodori_{ts}.wav"

print(f"🎙️ 合成中...")
print(f"   テキスト  : {TEXT}")
print(f"   キャプション: {CAPTION}\n")

request = SamplingRequest(
    text=TEXT,
    caption=CAPTION,
    no_ref=True,
    num_steps=NUM_STEPS,
    seed=SEED,
    cfg_scale_text=CFG_SCALE_TEXT,
    cfg_scale_caption=CFG_SCALE_CAPTION,
)

result = runtime.synthesize(request)

# 保存（soundfile で WAV 出力）
audio_np = result.audio.cpu().float().numpy()
if audio_np.ndim == 2:
    audio_np = audio_np.T  # (channels, samples) -> (samples, channels)
sf.write(output_path, audio_np, result.sample_rate)

print(f"✅ 保存: {output_path}")
display(Audio(output_path, autoplay=False))

In [ ]:
# ── VoiceDesign スタイル比較（複数のキャプションで同じテキストを読み上げ）──
COMPARE_TEXT = "今日は少し疲れましたが、お話できて嬉しいです。"

CAPTION_STYLES = [
    ("明るい女性",
     "明るく元気な若い女性の声で、笑顔が伝わるように読み上げてください。😊"),
    ("落ち着いた男性",
     "落ち着いた中年男性の声で、ゆっくりと丁寧に読み上げてください。"),
    ("疲れた声",
     "少し疲れた様子で、感情を込めてつぶやくように読み上げてください。😔"),
    ("アナウンサー",
     "NHKのアナウンサーのように、明瞭で標準的な発音で読み上げてください。"),
]

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

for style_name, caption in CAPTION_STYLES:
    print(f"\n🎙️ スタイル: {style_name}")
    print(f"   {caption}")

    output_path = f"{LOCAL_AUDIO_DIR}/style_{style_name}_{ts}.wav"

    request = SamplingRequest(
        text=COMPARE_TEXT,
        caption=caption,
        no_ref=True,
        num_steps=NUM_STEPS,
        seed=SEED,
        cfg_scale_text=CFG_SCALE_TEXT,
        cfg_scale_caption=CFG_SCALE_CAPTION,
    )

    result = runtime.synthesize(request)

    audio_np = result.audio.cpu().float().numpy()
    if audio_np.ndim == 2:
        audio_np = audio_np.T
    sf.write(output_path, audio_np, result.sample_rate)

    print(f"   ✅ 保存: {output_path}")
    display(Audio(output_path, autoplay=False))